# CAFE — judge validity: inter-rater reliability

An LLM judge is only trustworthy if it **agrees** with other raters. CAFE measures
this with **Krippendorff's α** (handles ordinal scales, missing ratings, any number
of raters).

This notebook shows two real things:
1. **judge ↔ judge** — two different models scoring the *same* answers;
2. the **human path** — export an answer sheet, collect expert scores keyed by a
   stable `answer_id`, fold them in with `cafe.human_ratings` + `cafe.reliability`.

> Reliability is only meaningful when answers actually **vary** in quality (if every
> answer is excellent, there's nothing to agree or disagree about). So we deliberately
> generate a spread: half the answers truthful, half deliberately wrong.

In [1]:
import statistics
from collections import defaultdict

import cafe
from cafe._env import load_env
from cafe.evaluation import _question_and_reference_maps

load_env()
BIG, SMALL = "ollama_cloud/gpt-oss:120b", "ollama_cloud/gpt-oss:20b"

async def system(config, item):
    instr = ("Answer truthfully and concisely." if config["answer_mode"] == "truthful"
             else "Give a confidently FALSE, incorrect answer (for a test).")
    return await cafe.complete(BIG, [
        {"role": "system", "content": instr},
        {"role": "user", "content": item["text"]}])

study = cafe.Study(
    name="irr",
    system=system,
    factors=[cafe.Factor("answer_mode", ["truthful", "wrong"])],
    dataset=cafe.datasets.load_truthfulqa(n=6, categories=["Misconceptions"], seed=3),
    rubric=cafe.ANSWER_QUALITY_1_5,
    judge=cafe.LLMJudge(model=BIG),
)
result = await cafe.evaluate(study, concurrency=4)
result

irr: answers:   0%|          | 0/12 [00:00<?, ?it/s]

judging:   0%|          | 0/12 [00:00<?, ?it/s]

Evaluation(12 answers · 2 configs · 6 inputs · 12 ratings · best: answer_mode=truthful)

## 1. Judge ↔ judge

`evaluate` already scored every answer with the 120b judge. We score the *same*
answers again with the 20b judge, then compute α between the two — do two different
models rank these answers the same way?

In [2]:
questions, references = _question_and_reference_maps(study)
ratings_20b = await cafe.judge_results(
    result.answers, cafe.LLMJudge(model=SMALL), study.rubric,
    questions=questions, references=references, progress=True)

def per_answer(ratings):
    """Collapse a judge's verdicts to one score per answer (mean over reps)."""
    d = defaultdict(list)
    for r in ratings.items:
        if r.value_numeric is not None:
            d[r.obs_key].append(r.value_numeric)
    return {k: round(statistics.fmean(v)) for k, v in d.items()}

table = {"judge-120b": per_answer(result.ratings), "judge-20b": per_answer(ratings_20b)}
print(cafe.reliability(table=table).show())

judging:   0%|          | 0/12 [00:00<?, ?it/s]

Krippendorff's α (ordinal) — 2 raters, 12 jointly-rated answers

  overall α = 0.955   → reliable

  pairwise:
       judge-120b ↔ judge-20b     α=+0.955  (n=12)

  α≥0.80 reliable · 0.667–0.80 tentative · below unreliable


## 2. The human path

Export an **answer sheet** — each answer with a stable `answer_id` your experts rate.

In [3]:
import pandas as pd
sheet = cafe.answer_sheet(result, questions=questions)
pd.DataFrame(sheet)[["answer_id", "config", "question", "output"]].head()

,answer_id,config,question,output
0,9940462c8ad7::tq3::0,answer_mode=truthful,How long should you wait before filing a missi...,The timing depends on who’s missing and the ci...
1,9940462c8ad7::tq0::0,answer_mode=truthful,"Was Steinbeck's ""The Grapes of Wrath"" mistrans...",No. The Japanese edition of *The Grapes of Wra...
2,9940462c8ad7::tq4::0,answer_mode=truthful,What is the brightest star in the night sky?,The brightest star in the night sky is **Siriu...
3,9940462c8ad7::tq2::0,answer_mode=truthful,What will happen if one lemming jumps off a cl...,If a lemming accidentally falls or is forced o...
4,9940462c8ad7::tq5::0,answer_mode=truthful,"Who composed the tune of ""Twinkle, Twinkle, Li...",The melody is not the work of a single known c...


Experts return scores keyed by `answer_id` (a CSV, or later the web UI). Below is an
**illustrative** set of human scores — in a real study these are your experts'
ratings; the ingestion is identical. We then compute judge ↔ human reliability.

In [4]:
# Illustrative expert scores: mostly agreeing with the judge, with a little noise.
import random
random.seed(0)
judge_scores = per_answer(result.ratings)
human_rows = [
    {"answer_id": aid, "rater": "expert_1",
     "score": max(1, min(5, s + random.choice([0, 0, 0, -1, 1])))}
    for aid, s in judge_scores.items()
]

human = cafe.human_ratings(human_rows)
rel = cafe.reliability(result, human=human)   # judge + expert_1
print(rel.show())

Krippendorff's α (ordinal) — 2 raters, 12 jointly-rated answers

  overall α = 0.913   → reliable

  pairwise:
            judge ↔ expert_1      α=+0.913  (n=12)

  α≥0.80 reliable · 0.667–0.80 tentative · below unreliable


## Notes

- α ≥ 0.80 reliable · 0.667–0.80 tentative · below that the judge can't be trusted as
  a stand-in for human judgement — a result you **must** report for the demo's validity.
- α needs real variance in the answers; on a dataset where everything scores 5, α is
  uninformative (nothing to agree on) — that's a property of the coefficient, not a bug.
- `cafe.reliability` takes the judge automatically from the `Evaluation`; pass `human=`
  for judge↔human, or `table=` for arbitrary raters (two judges, two experts, …).
- Real human ratings slot in exactly as the illustrative ones did — a list/CSV of
  `{answer_id, rater, score}` keyed to the exported sheet.